In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import argparse
import numpy as np
import random
import scipy
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
import random
from scipy.stats import kendalltau
import pandas as pd

In [2]:
edges=np.genfromtxt("./maayan-figeys/out.maayan-figeys",
                                        dtype=int)
edges=edges-1
adj = sp.coo_matrix((np.ones(edges.shape[0]), (edges[:, 0], edges[:, 1])),
                        shape=(2239, 2239))
adj = adj + adj.T.multiply(adj.T > adj) - adj.multiply(adj.T > adj)

adj_figeys = torch.FloatTensor(np.array(adj.todense()))
list_figeys=list(range(2239))

In [3]:
def min_max_normalization(tensor):
    # 找出数组的最小值和最大值
    min_val = torch.min(tensor)
    max_val = torch.max(tensor)

    # 进行最小 - 最大标准化
    normalized_tensor = (tensor - min_val) / (max_val - min_val)
    return normalized_tensor
# node_feature_BA = min_max_normalization(node_feature_BA)
# node_feature_BA1 = min_max_normalization(node_feature_BA1)

In [4]:
class GNN(torch.nn.Module):
    def __init__(self,input_feature,output_feature):
        super(GNN,self).__init__()
        self.w = nn.Parameter(torch.empty(size=(input_feature,output_feature)))
        self.a= nn.Parameter(torch.empty(size=(1,output_feature)))
        self.sigmod=torch.nn.Sigmoid()
        self.reset_parameters()
    def reset_parameters(self):
        #nn.init.xavier_uniform_(self.w.data,gain=1.414)
        #nn.init.xavier_uniform_(self.a.data,gain=1.414)
        for param in self.parameters():
             nn.init.xavier_uniform_(param)
        
    def forward(self,x,adj):
        adj=torch.Tensor(adj.numpy()+np.identity(adj.shape[0]))
        # adj=torch.FloatTensor(normalize_adj(sp.csr_matrix(adj)+ sp.eye(adj.shape[0])).todense())
        x=torch.mm(adj,x)
        x=torch.mm(x,self.w)
        x=x.add(self.a)
        x=torch.relu(x)
        return x


class CGNN(torch.nn.Module):
    def __init__(self):
        super(CGNN,self).__init__()
        self.layer2=GNN(48,6)
        self.layer3=GNN(6,12)
        # self.fc1=torch.nn.Linear(6,12)
        self.fc1=torch.nn.Linear(12,1)
        # self.fc2=torch.nn.Linear(24,1)
        
        
       
     
    
    
    def forward(self,x,adj,target_nodes):
        #x=self.layer1(x)
        # x=torch.cat([x1,x],dim=-1)
        # x=0.5*x+0.5*x1
        x=self.layer2(x,adj)
        x=self.layer3(x,adj)
       # x=self.layer4(x,adj)
        
        x=x[target_nodes]
        
        x=self.fc1(x.view(x.size(0),-1))
        x=torch.relu(x)
        # x=self.fc2(x)
        # x=torch.relu(x)
        x=x.flatten()
        
        return x

In [5]:
import optuna
import time
import numpy as np
import torch
import torch.optim as optim
from scipy.stats import kendalltau

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
# 假设你的模型和数据已定义
# from your_model import CGNN  # 导入你的CGNN模型
# 假设以下变量已提前定义：
# node_feature_figeys, adj_figeys, train_idx, list_figeys, list_pinggu
# train_figeys_lable, pinggu_true  # 训练标签和验证集真实值

def objective(trial):
    # 1. 定义超参数搜索空间
    learning_rate = trial.suggest_float(
        "learning_rate", 
        low=0.001, 
        high=0.01, 
        step=0.001  # 学习率：0.001-0.01，步长0.001
    )
    epochs = trial.suggest_int(
        "epochs", 
        low=100, 
        high=1000, # 训练次数：100-1000之间的任意整数
        step=100
    )


    epoch_1=trial.suggest_int(
        "epoch_1", 
        low=100, 
        high=500, # 训练次数：100-1000之间的任意整数
        step=100
    )
    learning_rate1 = trial.suggest_categorical(
        "learning_rate1", 
        [0.005,0.001,0.01]  # 仅允许这三个候选值
    )
    # 2. 初始化模型和优化器

    lable_figeys=np.load("lable/lable_figeys.npy")
    lable_figeys_t=torch.tensor(lable_figeys).float()
    node_feature_figeys=torch.load("EC做损失的特征向量/node_feature_figeys_LSTM_EC_"+str(learning_rate1)+"_"+str(epoch_1)+".pt")
    node_feature_figeys = min_max_normalization(node_feature_figeys)
    
    random.seed(17)
    np.random.seed(17)
    torch.manual_seed(17)
    
# 设置聚类数量
    k = 4
    X = node_feature_figeys.detach().numpy()

# 执行 KMeans 聚类
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X)

# labels 是每个节点对应的聚类类别，比如 labels[i] 是第i个节点所属的簇编号

# 将节点按簇分类整理
    cluster_nodes = {i: [] for i in range(k)}
    for idx, label in enumerate(labels):
        cluster_nodes[label].append(idx)
#选择训练集的节点
    n_nodes = node_feature_figeys.shape[0]

# 1. 随机采样 5% 的节点索引
    num_train = int(0.05 * n_nodes)+1
    num_pinggu = int(0.05 * n_nodes)+1

    train_idx=[]
    while len(train_idx) < num_train:
        for i in range(k):
            if cluster_nodes[i]:  # 确保当前簇中还有节点可取
                node = random.choice(cluster_nodes[i])  # 随机选一个
                train_idx.append(node)
                cluster_nodes[i].remove(node)
                if len(train_idx) >= num_train:
                    break

    pinggu_idx=[]
    while len(pinggu_idx) < num_train:
        for i in range(k):
            if cluster_nodes[i]:  # 确保当前簇中还有节点可取
                node = random.choice(cluster_nodes[i])  # 随机选一个
                pinggu_idx.append(node)
                cluster_nodes[i].remove(node)
                if len(pinggu_idx) >= num_train:
                    break

# 2. 构造训练集（只选取部分节点特征）
    train_figeys = node_feature_figeys[train_idx]
    train_figeys_lable = lable_figeys_t[train_idx]
# 3. 测试集就是整个 node_feature_BA
    
    pinggu_figeys = node_feature_figeys[pinggu_idx]
    pinggu_figeys_lable = lable_figeys_t[pinggu_idx]


    random.seed(17)
    np.random.seed(17)
    torch.manual_seed(17)
    # 2. 初始化模型和优化器
    model = CGNN()  # 实例化你的模型
    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=5e-4  # 保持权重衰减不变
    )
    
    # 3. 训练模型
    for epoch in range(epochs):
        # 训练过程（复用你的train函数逻辑）
        model.train()
        output = model(node_feature_figeys.data, adj_figeys, train_idx)
        loss_train = torch.nn.functional.mse_loss(output, train_figeys_lable)
        optimizer.zero_grad()
        loss_train.backward()
        optimizer.step()
        
        # 每100轮打印一次中间结果（可选）
        if (epoch + 1) % 100 == 0:
            print(f"当前参数组合: 学习率={learning_rate}, 训练次数={epochs}, 轮次={epoch+1}, 训练损失={loss_train.item():.4f}")
    
    # 4. 在验证集上评估RMSE
    model.eval()
    with torch.no_grad():  # 关闭梯度计算，节省内存
        pinggu_pre = model(node_feature_figeys.data, adj_figeys, pinggu_idx)
        # 转换为numpy数组计算RMSE
        pinggu_pre_np = pinggu_pre.detach().numpy()
        pinggu_true=pinggu_figeys_lable.detach().numpy()
        # print(kendalltau(pinggu_pre_np,pinggu_pre_np))
        rmse=kendalltau(pinggu_pre_np,pinggu_true).statistic
        # rmse = np.sqrt(np.mean((pinggu_true - pinggu_pre_np) **2))
    
    # 5. 返回需要最小化的RMSE（Optuna会寻找最小RMSE对应的参数）
    return rmse

if __name__ == '__main__':
    # 创建优化实例，方向为"最小化"RMSE
    study = optuna.create_study(direction="maximize")
    # 尝试20组参数组合
    study.optimize(objective, n_trials=200)
    
    # 输出最优结果
    print("\n优化完成！")
    print(f"最佳参数组合: {study.best_params}")
    print(f"最小RMSE: {study.best_value:.4f}")
    print(f"最佳试验编号: {study.best_trial.number}")

[I 2026-03-13 17:25:07,518] A new study created in memory with name: no-name-23cfbfce-d52a-40fd-b518-35610e1efaaa
[W 2026-03-13 17:25:07,542] Trial 0 failed with parameters: {'learning_rate': 0.004, 'epochs': 1000, 'epoch_1': 200, 'learning_rate1': 0.005} because of the following error: NameError("name 'KMeans' is not defined").
Traceback (most recent call last):
  File "C:\Users\Administrator\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_12632\1785217936.py", line 59, in objective
    kmeans = KMeans(n_clusters=k, random_state=42)
             ^^^^^^
NameError: name 'KMeans' is not defined
[W 2026-03-13 17:25:07,544] Trial 0 failed with value None.


NameError: name 'KMeans' is not defined

In [8]:
lable_figeys=np.load("lable/lable_figeys.npy")
lable_figeys_t=torch.tensor(lable_figeys).float()
node_feature_figeys=torch.load("EC做损失的特征向量/node_feature_figeys_LSTM_EC_0.001_200.pt")
node_feature_figeys = min_max_normalization(node_feature_figeys)

In [10]:
import numpy as np
from sklearn.cluster import KMeans
# 假设 node_feature_jazz 是你的节点特征矩阵，形状是 (num_nodes, feature_dim)
random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
    
# 设置聚类数量
k = 4
X = node_feature_figeys.detach().numpy()

# 执行 KMeans 聚类
kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(X)

# labels 是每个节点对应的聚类类别，比如 labels[i] 是第i个节点所属的簇编号

# 将节点按簇分类整理
cluster_nodes = {i: [] for i in range(k)}
for idx, label in enumerate(labels):
    cluster_nodes[label].append(idx)
#选择训练集的节点
n_nodes = node_feature_figeys.shape[0]

# 1. 随机采样 5% 的节点索引
num_train = int(0.05 * n_nodes)+1

train_idx=[]
while len(train_idx) < num_train:
    for i in range(k):
        if cluster_nodes[i]:  # 确保当前簇中还有节点可取
            node = random.choice(cluster_nodes[i])  # 随机选一个
            train_idx.append(node)
            cluster_nodes[i].remove(node)
            if len(train_idx) >= num_train:
                break

pinggu_idx=[]
while len(pinggu_idx) < num_train:
    for i in range(k):
        if cluster_nodes[i]:  # 确保当前簇中还有节点可取
            node = random.choice(cluster_nodes[i])  # 随机选一个
            pinggu_idx.append(node)
            cluster_nodes[i].remove(node)
            if len(pinggu_idx) >= num_train:
                break
print(train_idx)
print(pinggu_idx)
# 2. 构造训练集（只选取部分节点特征）
train_figeys = node_feature_figeys[train_idx]
train_figeys_lable = lable_figeys_t[train_idx]
# 3. 测试集就是整个 node_feature_BA
    
pinggu_figeys = node_feature_figeys[pinggu_idx]
pinggu_figeys_lable = lable_figeys_t[pinggu_idx]

[388, 2075, 425, 1050, 79, 823, 1217, 2119, 293, 1667, 1010, 797, 32, 2199, 24, 721, 123, 2089, 1191, 1207, 77, 2140, 697, 907, 266, 2125, 1068, 1178, 47, 1686, 75, 424, 572, 2059, 1720, 580, 50, 2167, 1110, 1526, 226, 2074, 1079, 613, 128, 1661, 162, 1879, 23, 1180, 579, 255, 212, 2233, 1646, 1365, 358, 1764, 195, 2054, 171, 2094, 733, 975, 4, 1486, 532, 1680, 28, 2190, 1248, 1045, 30, 1563, 538, 1711, 0, 2159, 583, 707, 72, 1703, 302, 732, 354, 1326, 65, 787, 56, 2062, 400, 574, 876, 2118, 860, 1568, 499, 2203, 174, 741, 167, 2220, 512, 938, 102, 1243, 124, 1732, 158, 1256, 8, 177]
[238, 172, 692, 817, 496, 1842, 63, 593, 200, 1053, 100, 1430, 9, 1824, 892, 1561, 94, 1426, 747, 1793, 494, 1407, 277, 588, 109, 761, 1144, 346, 83, 350, 1928, 1608, 131, 2103, 483, 946, 37, 799, 631, 290, 504, 2052, 476, 754, 441, 1438, 753, 240, 295, 516, 880, 1967, 116, 989, 865, 814, 336, 808, 1632, 893, 49, 1996, 770, 1804, 204, 2136, 772, 682, 1545, 1470, 835, 1932, 311, 481, 700, 670, 136, 1601, 48

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=9.
  warnings.warn(


In [12]:
import time
import numpy as np
import torch
import torch.optim as optim
from scipy.stats import kendalltau

# 确保随机种子完全一致
random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
# 对于CUDA环境，还需固定cuda随机种子
if torch.cuda.is_available():
    torch.cuda.manual_seed(17)
    torch.cuda.manual_seed_all(17)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train(epoch, model, optimizer, node_feature, adj, train_idx, train_label):
    model.train()
    output = model(node_feature.data, adj, train_idx)
    loss = torch.nn.functional.mse_loss(output, train_label)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

if __name__ == '__main__':
    # 确保数据与Optuna调优时完全一致
    # （需与objective函数中使用的node_feature_figeys、adj_figeys等保持相同）
    # 假设以下变量已正确定义：
    # node_feature_figeys, adj_figeys, train_idx, train_figeys_lable, sum_idx, sum_figeys_lable
    
    # 初始化模型和优化器（与Optuna中完全一致）
    model = CGNN()  # 重新初始化，确保权重初始状态一致
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.01,  # Optuna找到的最佳学习率
        weight_decay=5e-4
    )
    
    # 记录总训练时间
    t_total = time.time()
    loss_values = []
    
    # 训练600轮（与Optuna最佳参数一致）
    for epoch in range(300):
        loss = train(epoch, model, optimizer, node_feature_figeys, adj_figeys, train_idx, train_figeys_lable)
        loss_values.append(loss)
        
        # 每100轮打印训练损失（与Optuna中保持一致的监控频率）
        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}, 训练损失: {loss:.4f}")
    
    # 关键：添加与Optuna中完全一致的评估步骤
    model.eval()
    with torch.no_grad():
        # 使用相同的评估索引和标签
        pinggu_pre = model(node_feature_figeys.data, adj_figeys, pinggu_idx)
        pinggu_pre_np = pinggu_pre.detach().numpy()
        pinggu_true = pinggu_figeys_lable.detach().numpy()
        # 计算Kendall系数（与Optuna中评估指标一致）
        kendall = kendalltau(pinggu_pre_np, pinggu_true).statistic
        print(f"最终Kendall系数: {kendall:.6f}")
    
    print("Optimization Finished!")
    print(f"Total time elapsed: {time.time() - t_total:.2f}s")

Epoch 100, 训练损失: 0.6517
Epoch 200, 训练损失: 0.4068
Epoch 300, 训练损失: 0.3511
最终Kendall系数: 0.833041
Optimization Finished!
Total time elapsed: 28.24s


In [19]:
pinggu_pre = model(node_feature_figeys.data, adj_figeys, pinggu_idx)
# 转换为numpy数组计算RMSE
pinggu_pre_np = pinggu_pre.detach().numpy()
pinggu_true=pinggu_figeys_lable.detach().numpy()
# print(kendalltau(pinggu_pre_np,pinggu_pre_np))
rmse=kendalltau(pinggu_pre_np,pinggu_true).statistic
print(rmse)

0.8330407192114401


In [14]:
lable_figeys=np.load("lable/lable_figeys.npy")
lable_figeys_t=torch.tensor(lable_figeys).float()

In [16]:
model_cgnn=model
model_cgnn.eval()
start=time.time()
# output = model_cgnn(node_feature_figeys.data,adj_figeys,node_feature_figeys.data,node_feature_figeys.data)
output = model_cgnn(node_feature_figeys.data,adj_figeys,list_figeys)

end=time.time()
print(end-start)
output_rank=output.detach().numpy().argsort()

#print(output_rank)
#print(label_rank)
loss_train = torch.nn.functional.mse_loss(output,lable_figeys_t)
print(loss_train)
print(kendalltau(output.detach().numpy(),lable_figeys_t))
print(MI(output))

0.07502508163452148
tensor(0.1442, grad_fn=<MseLossBackward0>)
SignificanceResult(statistic=0.7064124643463905, pvalue=0.0)


NameError: name 'MI' is not defined

In [18]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

def calculate_node_ranking(node_scores):
    # 1. 处理纯得分列表（自动生成节点ID，如node_0, node_1...）
    if isinstance(node_scores, list) and all(isinstance(x, (int, float)) for x in node_scores):
        df = pd.DataFrame({
            'node': [f'node_{i}' for i in range(len(node_scores))],
            'score': node_scores
        })

    # 2. 按得分降序排序（影响力越高，得分越高，排名越靠前）
    df_sorted = df.sort_values(by='score', ascending=False).reset_index(drop=True)

    # 3. 计算稠密排名（同得分同排名）
    df_sorted['rank'] = df_sorted['score'].rank(method='dense', ascending=False).astype(int)

    return df_sorted

def calculate_ccdf_full_and_sample(df_ranked, N_total, step=40):

    # ========== 第一步：补全1~N_total所有位次的CCDF值 ==========
    # 1. 统计每个实际排名的节点数和累计数
    rank_count = df_ranked.groupby('rank')['node'].count().reset_index(name='n_i')
    rank_count['sum_n_i'] = rank_count['n_i'].cumsum()
    rank_count['ccdf'] = 1 - rank_count['sum_n_i'] / N_total

    # 2. 构建1~N_total的完整位次映射
    full_rank_list = list(range(1, N_total + 1))  # 1到网络总节点数的所有位次
    full_ccdf_list = []
    current_ccdf = 1.0  # 初始值
    rank_to_ccdf = dict(zip(rank_count['rank'], rank_count['ccdf']))

    # 补全每个位次的CCDF值（缺失位次继承前值）
    for r in full_rank_list:
        if r in rank_to_ccdf:
            current_ccdf = rank_to_ccdf[r]
        full_ccdf_list.append(current_ccdf)

    # 构建完整位次的DataFrame
    full_ccdf_df = pd.DataFrame({
        'rank': full_rank_list,  # 1~N_total的所有位次
        'ccdf': full_ccdf_list
    })

    # ========== 第二步：按步长抽取（基于完整位次） ==========
    total_full_ranks = len(full_ccdf_df)  # 等于N_total
    # 生成基础采样索引（步长为step）
    sample_indices = list(range(0, total_full_ranks, step))
    # 补充最后一个位次的索引
    last_index = total_full_ranks - 1
    if last_index not in sample_indices:
        sample_indices.append(last_index)
    # 去重+排序
    sample_indices = sorted(list(set(sample_indices)))

    # 按索引抽取数据
    sampled_ccdf_df = full_ccdf_df.iloc[sample_indices].reset_index(drop=True)
    sampled_ccdf_values = sampled_ccdf_df['ccdf'].tolist()
    sampled_ranks = sampled_ccdf_df['rank'].tolist()

    return full_ccdf_df, sampled_ccdf_df, sampled_ccdf_values, sampled_ranks

In [20]:
output=output.detach().numpy().tolist()
N_total = len(output)
print(N_total)
    # 步骤1：计算节点排名
df_ranked = calculate_node_ranking(output)
# print(df_ranked)
# total_ranks_actual = len(df_ranked.groupby('rank'))
total_ranks_actual= N_total
print(f"实际总排名数：{total_ranks_actual}")

# 步骤2：按步长40抽取CCDF（自动补充最后一个）
full_ccdf_df, sampled_ccdf_df, sampled_ccdf_values, sampled_ranks = calculate_ccdf_full_and_sample(df_ranked, N_total, step=1)

# 输出结果
# print("\n=== 抽取的位次（1~1200）===")


# for rank in sampled_ranks:
#     print(rank)

print(f"\n=== 最终抽取的CCDF值（共{len(sampled_ccdf_values)}个）===")
for value in sampled_ccdf_values:
    print(round(value, 4))  # 保留4位小数，更整洁
# print(sampled_ccdf_values)

2239
实际总排名数：2239

=== 最终抽取的CCDF值（共2239个）===
0.9996
0.9991
0.9987
0.9982
0.9978
0.9973
0.9969
0.9964
0.996
0.9955
0.9951
0.9946
0.9942
0.9937
0.9933
0.9929
0.9924
0.992
0.9915
0.9911
0.9906
0.9902
0.9897
0.9893
0.9888
0.9884
0.9879
0.9875
0.987
0.9866
0.9862
0.9857
0.9853
0.9848
0.9844
0.9839
0.9835
0.983
0.9826
0.9821
0.9817
0.9812
0.9808
0.9803
0.9799
0.9795
0.979
0.9786
0.9781
0.9777
0.9772
0.9768
0.9763
0.9759
0.9754
0.975
0.9745
0.9741
0.9736
0.9732
0.9728
0.9723
0.9719
0.9714
0.971
0.9705
0.9701
0.9696
0.9692
0.9687
0.9683
0.9678
0.9674
0.9669
0.9665
0.9661
0.9656
0.9652
0.9647
0.9643
0.9638
0.9634
0.9629
0.9625
0.962
0.9616
0.9611
0.9607
0.9603
0.9598
0.9594
0.9589
0.9585
0.958
0.9576
0.9571
0.9567
0.9562
0.9558
0.9553
0.9549
0.9544
0.954
0.9536
0.9531
0.9527
0.9522
0.9518
0.9513
0.9509
0.9504
0.95
0.9495
0.9491
0.9486
0.9482
0.9477
0.9473
0.9469
0.9464
0.946
0.9455
0.9451
0.9446
0.9442
0.9437
0.9433
0.9428
0.9424
0.9419
0.9415
0.941
0.9406
0.9402
0.9397
0.9393
0.9388
0.9384
0.93

In [22]:
model_cgnn=model
model_cgnn.eval()
start=time.time()
output = model_cgnn(node_feature_figeys.data,adj_figeys,list_figeys)
end=time.time()
print(end-start)
output_rank=output.detach().numpy().argsort()
temp=list(set(list_figeys)-set(train_idx+pinggu_idx))
# print(train_idx)
# print(pinggu_idx)
# print(temp)
output_temp=output[temp]
lable_figeys_t_temp=lable_figeys_t[temp]
print(kendalltau(output_temp.detach().numpy(),lable_figeys_t_temp))

print(kendalltau(output.detach().numpy(),lable_figeys_t))

for i in train_idx:
    output[i]=lable_figeys_t[i]
for i in pinggu_idx:
    output[i]=lable_figeys_t[i]
# print(output_rank)
# print(label_rank)
loss_train = torch.nn.functional.mse_loss(output,lable_figeys_t)
# print(loss_train)
print(kendalltau(output.detach().numpy(),lable_figeys_t))
print(MI(output))


0.10053372383117676
SignificanceResult(statistic=0.6917660780316907, pvalue=0.0)
SignificanceResult(statistic=0.7064124643463905, pvalue=0.0)
SignificanceResult(statistic=0.7220338132625443, pvalue=0.0)


NameError: name 'MI' is not defined

In [ ]:
def MI(res):
    a=pd.DataFrame(res.detach().numpy())
    x=len(a)
    a.rank(axis=0,method='min',numeric_only=None,
    na_option='keep',ascending=True,pct=False)
    b=a.iloc[:,0].value_counts()
    y=0
    for i in range(len(b)):
        y=y+b.iloc[i]*(b.iloc[i]-1)
    ans=(1-y/(x*(x-1)))*(1-y/(x*(x-1)))
    return ans

In [12]:
def calculate_top_k_jaccard(list1, list2,length):
    """
    计算两个列表按影响力排序后不同Top-K集合的Jaccard相似度
    
    参数:
    list1, list2: 节点影响力列表，下标为节点ID，值为影响力
    k_list: 需要计算的K值列表，默认为[100, 200, 300]
    
    返回:
    字典，键为K，值为对应Jaccard相似度
    """
    # 将列表转换为NumPy数组
    k_list = [int(length*0.05),int(length*0.06),int(length*0.07),int(length*0.08),int(length*0.09),int(length*0.10),int(length*0.11),int(length*0.12),int(length*0.13),int(length*0.14),int(length*0.15),int(length*0.16),int(length*0.17),int(length*0.18),int(length*0.19),int(length*0.20),int(length*0.21),int(length*0.22),int(length*0.23),int(length*0.24),int(length*0.25),int(length*0.26),int(length*0.27),int(length*0.28),int(length*0.29),int(length*0.30),int(length*0.31),int(length*0.32),int(length*0.33),int(length*0.34),int(length*0.35),int(length*0.36),int(length*0.37),int(length*0.38),int(length*0.39),int(length*0.40),int(length*0.41),int(length*0.42),int(length*0.43),int(length*0.44),int(length*0.45),int(length*0.46),int(length*0.47),int(length*0.48),int(length*0.49),int(length*0.50)]
    array1 = np.array(list1)
    array2 = np.array(list2)
    
    # 对数组排序并获取节点ID（argsort默认升序，加负号转为降序）
    sorted_indices1 = np.argsort(-array1)  # array1中影响力从大到小的节点ID
    # print(sorted_indices1)
    sorted_indices2 = np.argsort(-array2)  # array2中影响力从大到小的节点ID
    # print(sorted_indices2)
    # 计算各Top-K的Jaccard相似度
    results = {}
    for k in k_list:
        # 取前k个节点ID形成集合
        top_k_set1 = set(sorted_indices1[:k])
        top_k_set2 = set(sorted_indices2[:k])
        
        # 计算交集和并集大小
        intersection = len(top_k_set1 & top_k_set2)
        union = len(top_k_set1 | top_k_set2)
        
        # 计算Jaccard相似度
        jaccard = intersection / union if union > 0 else 0
        results[k] = jaccard
    
    return results

In [333]:
length=len(output)
k_list = [int(length*0.05),int(length*0.1),int(length*0.15),int(length*0.2),int(length*0.25),int(length*0.3),int(length*0.35),int(length*0.4),int(length*0.45),int(length*0.5)]
similarities = calculate_top_k_jaccard(output.detach().numpy(), lable_figeys_t,length)
for k, sim in similarities.items():
    print(sim)

0.574468085106383
0.6144578313253012
0.6335078534031413
0.6886792452830188
0.6890756302521008
0.6893939393939394
0.7024221453287197
0.7234726688102894
0.7067448680351907
0.6873315363881402
0.7135549872122762
0.7170263788968825
0.7117117117117117
0.7296137339055794
0.7346938775510204
0.7225433526011561
0.7279411764705882
0.7202797202797203
0.7219430485762144
0.7156549520766773
0.72
0.7168141592920354
0.7257142857142858
0.7221458046767538
0.716931216931217
0.7183098591549296
0.715698393077874
0.7047619047619048
0.6946039035591275
0.6986607142857143
0.7003257328990228
0.7148936170212766
0.7125129265770423
0.715438950554995
0.711764705882353
0.7244701348747592
0.7334593572778828
0.742354031510658
0.7427536231884058
0.7495559502664298
0.7482638888888888
0.7589743589743589
0.7665827036104114
0.7649958915365653
0.7622489959839357
0.7594339622641509


In [ ]:
# 最佳参数组合: {'learning_rate': 0.01, 'epochs': 300, 'epoch_1': 200, 'learning_rate1': 0.001}  figeys  4个聚簇
0.6660842628741939
0.6801027017727899
0.6889016437809943
0.7014148384521128
0.6950885563226102
0.7064124643463905
0.7094555545332334
0.7155937575344393
0.7225321364132679
0.7169918450433855
0.7318689126455458

In [ ]:
# {'learning_rate': 0.008, 'epochs': 495, 'epoch_1': 400, 'learning_rate1': 0.005}  5个聚簇
0.7109143117964398
0.6761115943756549
0.6378999827585335
0.5812807333939074
0.5247684615171132
0.48540185568462296
0.4481086436936275
0.4131818161888522
0.4026459867086868
0.3986300175375041
0.4030021228100379

In [ ]:
# 0.005/500 0.003/3000次 5个聚簇
#1.5β       0.49810053812187544
#1.2β   0.5935869589908415
#1.0β   0.610597962936673
#1.1β     0.6042784614395375
#1.3β     0.561394838308618
#1.4β     0.5235296194524466
#1.6β     0.464413770788594
#1.7β     0.4346831408549602
#1.8β     0.4269320807990303
#1.9β     0.41957454024546476
#2.0β     0.42246351526077386


In [ ]:
#0.005/100   0.01/3000  随机数为17 效果为 0.4925184617221323
#1.5β   0.005/200   0.01/300(300次之后都可以，后续看看1.0β可行不可行)    figeys的肯德尔系数0.485406089  6个聚簇       
#1.2β   0.6238593749
#1.0β   0.6920670161443011
#1.1β     0.6593890594
#1.3β     0.574527353
#1.4β     0.522188007
#1.6β     0.44877204172
#1.7β     0.4135871028
#1.8β     0.4037760889
#1.9β    0.40045963164818016
#2.0β   0.40532793772248915